In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
from adjustText import adjust_text
import os

In [ ]:
# add directory for plots if not already available
os.makedirs("../data/plots", exist_ok=True)

Here we will do some geographic analysis. Munich is subdivided into service areas for different police stations. We are interested in if there are differences between those.

In [ ]:
# read in all the csv files: we need Unfalltable (containing the accident type), unfalltext (containing the text) and the llm-predictions table
unfall_table = pd.read_csv("../data/D_Unfall_RData.csv")
unfalltext_table = pd.read_csv("../data/D_Unfalltext_image.csv")
llm_predictions_table = pd.read_csv("../data/few_shot_llm_predicted_labels_unfalltext_0_103111.csv")
# merge Unfalltable and Unfalltexttable
unfall_text_table_labels_merged = unfall_table.merge(unfalltext_table, on="UN_KEY", how="right")[["US_GEO_X", "US_GEO_Y", "Unfalltyp", "UN_KEY"]].dropna()

In [ ]:
# merge table with the llm prediciton
unfall_text_table_labels_merged = unfall_text_table_labels_merged.merge(llm_predictions_table, on="UN_KEY", how="left")[["US_GEO_X", "US_GEO_Y", "Unfalltyp", "few_shot_LLM_Labels"]]

In [ ]:
# read in shape-file indicating police subdivisions
stations = gpd.read_file("../explorative/police_map.geojson")

In [ ]:
# define geometry and convert table into geopandas-Dataframe
geometry = [Point(xy) for xy in zip(unfall_text_table_labels_merged['US_GEO_X'], unfall_text_table_labels_merged['US_GEO_Y'])]
points = gpd.GeoDataFrame(unfall_text_table_labels_merged, geometry=geometry, crs=stations.crs)

In [ ]:
# perform geopandas-join
points_in_areas = gpd.sjoin(points, stations, how="left", predicate="within")
points_in_areas = points_in_areas.dropna(subset=['index_right'])

In [ ]:
# for each area how many accidents in total:
counts = points_in_areas.groupby('index_right').size().rename('total_count')
# for each area how many were labeled as 7 by human:
class_7_counts = points_in_areas[points_in_areas['Unfalltyp'] == 7].groupby('index_right').size().rename('class_7_count')
# for each area how many were predicted as 7 by the llm few-shot:
llm_class_7_counts = points_in_areas[points_in_areas['few_shot_LLM_Labels'] == 7].groupby('index_right').size().rename('llm_class_7_counts')
# for each area, among those human-labeled as 7, for how many did llm agree:
llm_agrees_with_human_7 = points_in_areas[(points_in_areas['Unfalltyp'] == 7) & (points_in_areas['few_shot_LLM_Labels'] == 7)].groupby('index_right').size().rename('llm_agrees_with_human_7')
# for each area, how big was the overall compliance between llm and human
llm_total_agreement = points_in_areas[points_in_areas['Unfalltyp'] == points_in_areas['few_shot_LLM_Labels']].groupby('index_right').size().rename('llm_total_agreement')

In [ ]:
# join all columns together
stations = stations.join(counts).join(class_7_counts).join(llm_class_7_counts).join(llm_agrees_with_human_7).join(llm_total_agreement)

In [ ]:
# percentage of human-labeled 7
stations['class_7_ratio'] = stations['class_7_count'] / stations['total_count']
# percentage of llm-labeled 7
stations['llm_class_7_ratio'] = stations['llm_class_7_counts'] / stations['total_count']
# percentage of llm-labeled 7 among human-labeled 7
stations['llm_human_class_7_agreement_ratio'] = stations['llm_agrees_with_human_7'] / stations['class_7_count']
# percentage of overall agreeeing labels among all labels
stations['llm_total_agreement_ratio'] = stations['llm_total_agreement'] / stations['total_count']

In [ ]:
# remove NAs values and areas with less than 300 observations
stations_cleaned = stations.dropna(subset=['total_count'])
stations_cleaned = stations_cleaned[stations_cleaned['total_count'] >= 300]

stations_cleaned['class_7_count'].fillna(0, inplace=True)
stations_cleaned['llm_class_7_counts'].fillna(0, inplace=True)
stations_cleaned['llm_agrees_with_human_7'].fillna(0, inplace=True)
stations_cleaned['llm_total_agreement'].fillna(0, inplace=True)

stations_cleaned['class_7_ratio'].fillna(0, inplace=True)
stations_cleaned['llm_class_7_ratio'].fillna(0, inplace=True)
stations_cleaned['llm_human_class_7_agreement_ratio'].fillna(0, inplace=True)
stations_cleaned['llm_total_agreement_ratio'].fillna(0, inplace=True)

# make names more readable
stations_cleaned['name'] = stations_cleaned['name'].str.rsplit('- ', n=1).str[1]

Now we can plot the type 7 ratio in all police service areas.

In [ ]:
# first we plot with the names of the police districts
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# calculate range of ratio for color range. We take the minimal value from the other evaluation point (llm-prediction) which is much lower. The reason we do this is 
# because we want to compare both plots
vmin = stations_cleaned["llm_class_7_ratio"].min()
vmax = stations_cleaned["class_7_ratio"].max()

stations_cleaned.plot(column='class_7_ratio', cmap='OrRd', legend=False, ax=ax, edgecolor='black', vmin=vmin, vmax=vmax)

# adjust some positions and texts
for idx, row in stations_cleaned.iterrows():
    x, y = row.geometry.centroid.x, row.geometry.centroid.y
    text = f"{row['name']} \n {round(row['class_7_ratio'] * 100)}% of {int(row['total_count'])}"
    if row['name'] in "Hauptbahnhof":
        ax.text(x - 0.03, y - 0.0, text, fontsize=7, ha='left', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    elif row['name'] in "Westend":
        ax.text(x - 0.015, y - 0.004, text, fontsize=7, ha='left', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    elif row['name'] in "Altstadt":
        ax.text(x - 0.001, y - 0.003, text, fontsize=7, ha='left', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    else:
        ax.text(x, y, text, fontsize=7, ha='center', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

# need neither coordinates nor ticks
ax.set_xlabel('')
ax.set_ylabel('')

ax.set_xticks([])
ax.set_yticks([])
ax.set_xticklabels([])
ax.set_yticklabels([])

plt.title("Ratio of type 7 accidents for each police territory")
cbar = fig.colorbar(ax.collections[0], ax=ax, shrink=0.5, fraction=0.02, pad=0.04, format="%.2f")

plt.savefig("../data/plots/human_7_amount_map_labeled_absolute.png", dpi=300, bbox_inches='tight')

plt.title(None)
plt.savefig("../data/plots/human_7_amount_map_labeled_absolute_no_title.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Same plot as above but with its own color range.
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

stations_cleaned.plot(column='class_7_ratio', cmap='OrRd', legend=False, ax=ax, edgecolor='black')

for idx, row in stations_cleaned.iterrows():
    x, y = row.geometry.centroid.x, row.geometry.centroid.y
    text = f"{row['name']} \n {round(row['class_7_ratio'] * 100)}% of {int(row['total_count'])}"
    if row['name'] in "Hauptbahnhof":
        ax.text(x - 0.03, y - 0.0, text, fontsize=7, ha='left', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    elif row['name'] in "Westend":
        ax.text(x - 0.015, y - 0.004, text, fontsize=7, ha='left', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    elif row['name'] in "Altstadt":
        ax.text(x - 0.001, y - 0.003, text, fontsize=7, ha='left', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    else:
        ax.text(x, y, text, fontsize=7, ha='center', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

ax.set_xlabel('')
ax.set_ylabel('')

ax.set_xticks([])
ax.set_yticks([])
ax.set_xticklabels([])
ax.set_yticklabels([])

plt.title("Ratio of type 7 accidents for each police territory")
cbar = fig.colorbar(ax.collections[0], ax=ax, shrink=0.5, fraction=0.02, pad=0.04, format="%.2f")

plt.savefig("../data/plots/human_7_amount_map_labeled_relative.png", dpi=300, bbox_inches='tight')

plt.title(None)

plt.savefig("../data/plots/human_7_amount_map_labeled_relative_no_title.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# we also plot the agreement ratio between llm and human, independent from each class. Beware though that this is just showing the case 3 from the few-shot model, i.e. 
# if there is an area with a hight type 7 ratio, naturally the agreement with the llm will be lower there, since it predicts a lot of 7s as 5s.
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
stations_cleaned.plot(column='llm_total_agreement_ratio', cmap='Blues', legend=False, ax=ax, edgecolor='black')

for idx, row in stations_cleaned.iterrows():
    x, y = row.geometry.centroid.x, row.geometry.centroid.y
    text = f"{row['name']} \n {round(row['llm_total_agreement_ratio'] * 100)}% of {int(row['total_count'])}"
    if row['name'] in "Hauptbahnhof":
        ax.text(x - 0.03, y - 0.0, text, fontsize=7, ha='left', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    elif row['name'] in "Westend":
        ax.text(x - 0.015, y - 0.004, text, fontsize=7, ha='left', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    elif row['name'] in "Altstadt":
        ax.text(x - 0.001, y - 0.003, text, fontsize=7, ha='left', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    else:
        ax.text(x, y, text, fontsize=7, ha='center', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

ax.set_xlabel('')
ax.set_ylabel('')

ax.set_xticks([])
ax.set_yticks([])
ax.set_xticklabels([])
ax.set_yticklabels([])

plt.title("Agreement ratio of LLM-Few-Shot and human judgement")
cbar = fig.colorbar(ax.collections[0], ax=ax, shrink=0.5, fraction=0.02, pad=0.04, format="%.2f")
plt.savefig("../data/plots/agreement_map_labeled.png", dpi=300, bbox_inches='tight')

plt.title(None)
plt.savefig("../data/plots/agreement_map_labeled_no_title.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# we can also plot the agreement ratio, given that the human predicted 7
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
stations_cleaned.plot(column='llm_human_class_7_agreement_ratio', cmap='Blues', legend=False, ax=ax, edgecolor='black')

for idx, row in stations_cleaned.iterrows():
    x, y = row.geometry.centroid.x, row.geometry.centroid.y
    text = f"{row['name']} \n {round(row['llm_human_class_7_agreement_ratio'] * 100)}% of {int(row['class_7_count'])}"
    if row['name'] in "Hauptbahnhof":
        ax.text(x - 0.03, y - 0.0, text, fontsize=7, ha='left', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    elif row['name'] in "Westend":
        ax.text(x - 0.015, y - 0.004, text, fontsize=7, ha='left', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    elif row['name'] in "Altstadt":
        ax.text(x - 0.001, y - 0.003, text, fontsize=7, ha='left', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    else:
        ax.text(x, y, text, fontsize=7, ha='center', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

ax.set_xlabel('')
ax.set_ylabel('')

ax.set_xticks([])
ax.set_yticks([])
ax.set_xticklabels([])
ax.set_yticklabels([])

plt.title("Agreement ratio by LLM-Few-Shot of human-labeled class 7")
cbar = fig.colorbar(ax.collections[0], ax=ax, shrink=0.5, fraction=0.02, pad=0.04, format="%.2f")
plt.savefig("../data/plots/agreement_among_7_map_labeled.png", dpi=300, bbox_inches='tight')

plt.title(None)
plt.savefig("../data/plots/agreement_among_7_map_labeled_no_title.png", dpi=300, bbox_inches='tight')

In [ ]:
# here we plot the type 7 ratio according to LLM-few-shot but on the same scale as the human-predicted plot.
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# so same color range as above here:
vmin = stations_cleaned["llm_class_7_ratio"].min()
vmax = stations_cleaned["class_7_ratio"].max()

stations_cleaned.plot(column='llm_class_7_ratio', cmap='OrRd', legend=False, ax=ax, edgecolor='black', vmin=vmin, vmax=vmax)

for idx, row in stations_cleaned.iterrows():
    x, y = row.geometry.centroid.x, row.geometry.centroid.y
    text = f"{row['name']} \n {round(row['llm_class_7_ratio'] * 100)}% of {int(row['total_count'])}"
    if row['name'] in "Westend":
        ax.text(x - 0.015, y - 0.004, text, fontsize=7, ha='left', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    elif row['name'] in "Altstadt":
        ax.text(x - 0.001, y - 0.003, text, fontsize=7, ha='left', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    else:
        ax.text(x, y, text, fontsize=7, ha='center', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

ax.set_xlabel('')
ax.set_ylabel('')

ax.set_xticks([])
ax.set_yticks([])
ax.set_xticklabels([])
ax.set_yticklabels([])

plt.title("Ratio of type 7 accidents for each police territory as predicted by LLM-Few-Shot")
cbar = fig.colorbar(ax.collections[0], ax=ax, shrink=0.5, fraction=0.02, pad=0.04, format="%.2f")
plt.savefig("../data/plots/llm_7_amount_map_labeled_absolute.png", dpi=300, bbox_inches='tight')

plt.title(None)
plt.savefig("../data/plots/llm_7_amount_map_labeled_absolute_no_title.png", dpi=300, bbox_inches='tight')

In [ ]:
# and again the plot for the type 7 as predicted by the LLM but now on its own color range
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

stations_cleaned.plot(column='llm_class_7_ratio', cmap='OrRd', legend=False, ax=ax, edgecolor='black')

for idx, row in stations_cleaned.iterrows():
    x, y = row.geometry.centroid.x, row.geometry.centroid.y
    text = f"{row['name']} \n {round(row['llm_class_7_ratio'] * 100)}% of {int(row['total_count'])}"
    if row['name'] in "Westend":
        ax.text(x - 0.015, y - 0.004, text, fontsize=7, ha='left', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    elif row['name'] in "Altstadt":
        ax.text(x - 0.001, y - 0.003, text, fontsize=7, ha='left', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    else:
        ax.text(x, y, text, fontsize=7, ha='center', color='black', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

ax.set_xlabel('')
ax.set_ylabel('')

ax.set_xticks([])
ax.set_yticks([])
ax.set_xticklabels([])
ax.set_yticklabels([])

plt.title("Ratio of type 7 accidents for each police territory as predicted by LLM-Few-Shot")
cbar = fig.colorbar(ax.collections[0], ax=ax, shrink=0.5, fraction=0.02, pad=0.04, format="%.2f")
plt.savefig("../data/plots/llm_7_amount_map_labeled_relative.png", dpi=300, bbox_inches='tight')

plt.title(None)
plt.savefig("../data/plots/llm_7_amount_map_labeled_relative_no_title.png", dpi=300, bbox_inches='tight')

In [ ]:
# now we can combine the plots for human-predicted type 7 ratio and llm-predicted type 7 ratio into a single scatterplots
fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(stations_cleaned['class_7_ratio'], stations_cleaned['llm_class_7_ratio'], alpha=0.7)
ax.set_xlabel("Class 7 Ratio (Human)")
ax.set_ylabel("Class 7 Ratio (LLM)")
ax.set_title("Class 7 Ratio: Human vs. LLM")

for i, row in stations_cleaned.iterrows():
    ax.text(row['class_7_ratio'] + 0.003, row['llm_class_7_ratio'], row['name'], fontsize=8, alpha=0.7, color='black')

plt.tight_layout()
plt.savefig("../data/plots/scatterplot.png", dpi=300, bbox_inches='tight')
plt.title(None)
plt.savefig("../data/plots/scatterplot_no_title.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8), constrained_layout=True)

vmin = stations_cleaned["llm_class_7_ratio"].min()
vmax = stations_cleaned["class_7_ratio"].max()

# First plot (class_7_ratio)
stations_cleaned.plot(column='class_7_ratio', cmap='OrRd', legend=False, ax=axes[0], edgecolor='black', vmin=vmin, vmax=vmax)

# Second plot (llm_class_7_ratio)
stations_cleaned.plot(column='llm_class_7_ratio', cmap='OrRd', legend=False, ax=axes[1], edgecolor='black', vmin=vmin, vmax=vmax)

# Formatting both plots
for ax in axes:
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xticklabels([])
    ax.set_yticklabels([])

# Add a single colorbar for both plots
cbar = fig.colorbar(axes[0].collections[0], ax=axes, orientation="horizontal", shrink=0.5, fraction=0.05, pad=0.05, format="%.2f")

plt.savefig("../data/plots/geoplots_combined.png", dpi=300, bbox_inches='tight')
plt.show()